In [1]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [2]:
dataset = pd.read_csv("data_treated.csv")

treatment_variables = []
with open("diagnosis.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

diagnosis_variables = []
with open("treatment.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [3]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
# Since every duplicated patient has a consistent event indicator (regional ln recurrence) I can use this method
groups = survival_dataset['PERSON_ID']
event_indicator = survival_dataset['Regional LN recurrence']

In [4]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=event_indicator, groups=groups)

train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]
groups = groups.iloc[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=12)

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 500],  # 100 can already be stable; 500 is often enough
    "min_samples_split": [2, 5],      # Prevent overfitting; 10+ rarely helps in clinical data
    "min_samples_leaf": [1, 5, 10],   # Key for balancing bias-variance
    "max_features": ["sqrt", 0.5],    # sqrt is typical; 0.5 can boost diversity
    "max_depth": [None, 10],          # None (unlimited) vs limited depth
    "bootstrap": [True]               # Keep True — RSF depends on bootstrapping
}

# {'bootstrap': True, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 200}
# Best C-index: 0.7380676257590892

sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf.split(X=x_train, y=event_indicator, groups=groups)
rsf = RandomSurvivalForest(n_jobs=-1, random_state=12)

best_model = RandomSurvivalForest(
    bootstrap = True, 
    max_depth = 10, max_features = 'sqrt', 
    min_samples_leaf = 10,
    min_samples_split = 2, 
    n_estimators = 200,
    random_state=12)

In [7]:
grid_search = GridSearchCV(
    estimator=rsf,
    param_grid=param_grid,
    cv=cv,
    scoring=score_survival_model,
    n_jobs=-1,
    verbose=1
)

In [8]:
grid_search.fit(data_x, data_y)

Fitting 4 folds for each of 72 candidates, totalling 288 fits


GridSearchCV(cv=<generator object _BaseKFold.split at 0x000002583D9DAB00>,
             estimator=RandomSurvivalForest(n_jobs=-1, random_state=12),
             n_jobs=-1,
             param_grid={'bootstrap': [True], 'max_depth': [None, 10],
                         'max_features': ['sqrt', 0.5],
                         'min_samples_leaf': [1, 5, 10],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200, 500]},
             scoring=<function score_survival_model at 0x000002580BA76200>,
             verbose=1)

In [9]:
best_parameters = grid_search.best_params_
best_c_index = grid_search.best_score_
best_model = grid_search.best_estimator_
print("Best parameters:", best_parameters)
print("Best C-index:", best_c_index)

Best parameters: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 200}
Best C-index: 0.8059983491514462


In [7]:
results = train_evaluate(data_x, data_y, groups, sgkf, best_model)
results

,fold,c_index,c_ipcw,brier_percentile,brier_alltimes,brier_fixed
0,0,0.729718,0.567355,0.017211,0.036589,0.028014
1,1,0.781692,0.045275,0.007684,0.190040,0.022534
2,2,0.830201,0.748291,0.009085,0.016494,0.013873
3,3,0.783567,0.033095,0.010530,0.024240,0.012430
4,mean,0.781295,0.348504,0.011127,0.066841,0.019213
5,std,0.041058,0.364763,0.004219,0.082548,0.007371
